# SAMSum Pegasus: train, resume and evaluate

Select **Runtime > Change runtime type > GPU**. Save this notebook to Drive once, then rerun setup after a runtime reset; no source or dataset uploads are needed. The repository must contain the pushed corrections. Full training may exceed a free session. Checkpoints and reports are written to Drive, and only saved progress can be resumed.

The completed T4 report in the repository records 45.26 ROUGE-1, 35.60 ROUGE-L, and 0.792s mean warm pipeline latency across 819 test examples. These are recorded results, not guaranteed outcomes for a new run. API/network latency is excluded.


In [ ]:
from pathlib import Path
import os, subprocess, sys
project = Path('/content/textsummarizer')
if not project.exists():
    subprocess.run(['git', 'clone', 'https://github.com/GuptaVardan/Text-Summarizer.git', str(project)], check=True)
elif not (project / '.git').exists():
    raise RuntimeError('A previous ZIP checkout exists. Use a fresh runtime for this Git-based notebook; preserve Drive artifacts first.')
os.chdir(project)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-dev.txt'], check=True)
print('Source revision:', subprocess.check_output(['git','rev-parse','HEAD'], text=True).strip())
# If Colab requests a restart after installation, restart, then rerun this cell.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import torch, yaml, json
from importlib.metadata import version
assert torch.cuda.is_available(), 'Select a GPU runtime before full Pegasus training.'
print('GPU:', torch.cuda.get_device_name(0))
# Keep this path unchanged to resume the same run. Use a different folder for a new experiment.
run_root = Path('/content/drive/MyDrive/textsummarizer-verification')
run_root.mkdir(parents=True, exist_ok=True)
config = yaml.safe_load(Path('config/config.yaml').read_text())
config['model_trainer']['root_dir'] = str(run_root / 'model_trainer')
config['model_evaluation'].update({
    'model_path': str(run_root / 'model_trainer/pegasus-samsum-model'),
    'tokenizer_path': str(run_root / 'model_trainer/tokenizer'),
})
Path('config/config.yaml').write_text(yaml.safe_dump(config))
# Existing runs must retain their original data, hyperparameters and package versions.
print({name: version(name) for name in ('torch','transformers','datasets','accelerate')})


In [ ]:
# Restore local data after a reset. Training is not launched by this cell.
from src.textSummarizer.pipeline.stage_1_data_ingestion_pipeline import DataIngestionTrainingPipeline
from src.textSummarizer.pipeline.stage_2_data_transformation_pipeline import DataTransformationTrainingPipeline
DataIngestionTrainingPipeline().initiate_data_ingestion()
DataTransformationTrainingPipeline().initiate_data_transformation()
from datasets import load_from_disk
ds = load_from_disk(config['model_trainer']['data_path'])
print({split: len(data) for split, data in ds.items()})
assert {split: len(data) for split, data in ds.items()} == {'train':14731, 'validation':818, 'test':819}


In [ ]:
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], check=True)


## Train or resume

This cell selects the latest checkpoint when available and passes it to Trainer to restore saved state. If final model artifacts already exist, it skips training. A reset before the first checkpoint at step 250 loses all unsaved training progress. If a checkpoint save was interrupted, inspect it and select the previous complete checkpoint instead. Keep dependency versions and all training settings consistent across resumed sessions.


In [ ]:
from transformers.trainer_utils import get_last_checkpoint
from src.textSummarizer.config.configuration import ConfigurationManager
from src.textSummarizer.components.model_trainer import ModelTrainer
trainer_root = run_root / 'model_trainer'
final_model = trainer_root / 'pegasus-samsum-model'
if (final_model / 'config.json').exists() and (trainer_root / 'tokenizer/tokenizer_config.json').exists():
    print('Final model/tokenizer found. Training skipped; continue to evaluation.')
else:
    checkpoint = get_last_checkpoint(str(trainer_root)) if trainer_root.exists() else None
    if checkpoint:
        for name in ('trainer_state.json','optimizer.pt','scheduler.pt'):
            assert (Path(checkpoint) / name).exists(), f'Incomplete checkpoint: {name}'
        print('Resuming:', checkpoint)
    else:
        (run_root / 'source_revision.txt').write_text(subprocess.check_output(['git','rev-parse','HEAD'], text=True))
        (run_root / 'config.yaml').write_text(Path('config/config.yaml').read_text())
        (run_root / 'params.yaml').write_text(Path('params.yaml').read_text())
        (run_root / 'package_versions.json').write_text(json.dumps({n:version(n) for n in ('torch','transformers','datasets','accelerate','rouge-score')}, indent=2))
        print('Starting a fresh run.')
    ModelTrainer(ConfigurationManager().get_model_trainer_config()).train(resume_from_checkpoint=checkpoint)


## Smoke benchmark and full-test evaluation

The smoke report covers ten seeded test examples; it is not an overall quality claim. The full report covers all 819 test examples. Progress prints every 25 examples. Wait for the command to return successfully and inspect saved predictions as well as aggregate scores. Timings include tokenization, generation and decoding after warmups; HTTP and model loading are excluded.


In [ ]:
common = [sys.executable, 'evaluate_model.py',
    '--model-path', str(trainer_root / 'pegasus-samsum-model'),
    '--tokenizer-path', str(trainer_root / 'tokenizer')]
subprocess.run(common + ['--sample-size','10','--output',str(run_root / 'smoke')], check=True)


In [ ]:
subprocess.run(common + ['--output',str(run_root / 'benchmark')], check=True)
print((run_root / 'benchmark/metrics.json').read_text())


Retain the checkpoint, training logs, configuration, source revision, package versions, metrics and `predictions.jsonl`. The JSON summary alone cannot prove training provenance. After a runtime reset, reconnect to a GPU and rerun this notebook's cells; intermediate checkpoints in the same Drive folder permit resumption. Reconnecting a browser does not necessarily mean the runtime stopped: check for an active job before launching another.

To infer on a CPU laptop, download both final model and tokenizer folders into `artifacts/model_trainer/`. The T4 latency report does not describe laptop or HTTP response times. Historical notebooks in `research/` contain earlier split/evaluation bugs and are not the reproduction entry point.
